# Silver to Gold Star Schema

This notebook rebuilds the Gold dimensional model from the validated Silver layer.

## Outputs

| Table | Grain | Write strategy |
|---|---|---|
| `dim_date` | One row per calendar date | Delta overwrite |
| `dim_product` | One row per product | Delta overwrite |
| `dim_store` | One row per store | Delta overwrite |
| `fact_sales` | One row per sale transaction | Delta overwrite |
| `fact_inventory_position` | One row per supplied store-product position | Delta overwrite |

The stable source business keys `product_id` and `store_id` are used directly. `dim_date` uses an integer `date_key` in `yyyyMMdd` format. Because every run reads the complete, conformed Silver state, Gold is deterministically rebuilt with overwrite semantics.

> `fact_inventory_position` represents the currently supplied inventory position. The source has no business snapshot date, so this table must not be described as historical inventory. Missing store-product combinations remain missing and are not converted to zero stock.


## 1. Setup

Import PySpark functions and provide small validation helpers. \
Blocking checks raise an exception before the affected tables are written.

In [ ]:
from pyspark.sql import functions as F

def require(condition, message):
    if not condition:
        raise ValueError(message)

def validate_unique_key(df, table_name, key_columns):
    row_count = df.count()
    require(row_count > 0, f"{table_name} is empty.")

    null_condition = F.lit(False)
    for column_name in key_columns:
        null_condition = null_condition | F.col(column_name).isNull()

    null_key_count = df.filter(null_condition).count()
    distinct_key_count = df.select(*key_columns).distinct().count()

    require(null_key_count == 0, f"{table_name} contains null keys.")
    require(row_count == distinct_key_count, f"{table_name} contains duplicate keys.")
    return row_count

def overwrite_delta_table(df, table_name):
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(table_name)
    )
    print(f"{table_name} written successfully.")

## 2. Read the Silver layer

Read the five complete Silver Delta tables from the default Lakehouse and display their current row counts.

In [ ]:
silver_products_df = spark.table("silver_products")
silver_stores_df = spark.table("silver_stores")
silver_inventory_df = spark.table("silver_inventory_position")
silver_calendar_df = spark.table("silver_calendar")
silver_sales_df = spark.table("silver_sales")

silver_counts = [
    ("silver_products", silver_products_df.count()),
    ("silver_stores", silver_stores_df.count()),
    ("silver_inventory_position", silver_inventory_df.count()),
    ("silver_calendar", silver_calendar_df.count()),
    ("silver_sales", silver_sales_df.count()),
]
display(spark.createDataFrame(silver_counts, ["table_name", "row_count"]))

## 3. Build dimensions

### 3.1 Date dimension

Create reporting attributes from `calendar_date`. `month_number` supports chronological sorting of `month_name`; `year_month` supports trends spanning multiple years. ISO weekday numbering uses Monday as 1 and Sunday as 7.

In [ ]:
dim_date_df = (
    silver_calendar_df
    .select(
        F.date_format("calendar_date", "yyyyMMdd").cast("int").alias("date_key"),
        F.col("calendar_date"),
        F.year("calendar_date").alias("year"),
        F.quarter("calendar_date").alias("quarter_number"),
        F.month("calendar_date").alias("month_number"),
        F.date_format("calendar_date", "MMMM").alias("month_name"),
        F.date_format("calendar_date", "yyyy-MM").alias("year_month"),
        F.dayofmonth("calendar_date").alias("day_of_month"),
        (((F.dayofweek("calendar_date") + F.lit(5)) % F.lit(7)) + F.lit(1))
        .cast("int")
        .alias("day_of_week_number"),
        F.date_format("calendar_date", "EEEE").alias("day_name"),
        F.dayofweek("calendar_date").isin(1, 7).alias("is_weekend"),
    )
    # Monday of the calendar date's ISO week
    .withColumn(
        "week_start_date",
        F.date_sub(
            F.col("calendar_date"),
            F.col("day_of_week_number") - F.lit(1),
        ),
    )
    # ISO week-year is determined by the Thursday in that week
    .withColumn(
        "_iso_week_year",
        F.year(
            F.date_add(
                F.col("week_start_date"),
                3,
            )
        ),
    )

    # Examples: 2023-W01, 2023-W39
    .withColumn(
        "year_week",
        F.concat(
            F.col("_iso_week_year").cast("string"),
            F.lit("-W"),
            F.lpad(
                F.weekofyear("calendar_date").cast("string"),
                2,
                "0",
            ),
        ),
    )

    .drop("_iso_week_year")
)

display(dim_date_df.orderBy("calendar_date").limit(10))

### 3.2 Product dimension

Retain descriptive product attributes and add unit margin measures. A zero retail price produces a null margin percentage rather than a divide-by-zero error.

In [ ]:
dim_product_df = (
    silver_products_df
    .select(
        "product_id",
        "product_name",
        "product_category",
        F.col("product_cost").cast("decimal(18,2)").alias("product_cost"),
        F.col("product_price").cast("decimal(18,2)").alias("product_price"),
    )
    .withColumn(
        "unit_margin",
        (F.col("product_price") - F.col("product_cost")).cast("decimal(18,2)"),
    )
    .withColumn(
        "unit_margin_pct",
        F.when(
            F.col("product_price") != 0,
            (F.col("unit_margin") / F.col("product_price")).cast("decimal(18,4)"),
        ),
    )
)

display(dim_product_df.orderBy("product_id"))

### 3.3 Store dimension

Retain stable store attributes. A changing `store_age` value is intentionally not persisted because it depends on the reporting date.

In [ ]:
dim_store_df = silver_stores_df.select(
    "store_id",
    "store_name",
    "store_city",
    "store_location",
    "store_open_date",
)

display(dim_store_df.orderBy("store_id").limit(10))

## 3.4 Validate dimensions

Validate that each Gold dimension is non-empty and contains a unique, non-null key at its declared grain.

Source-level data-quality checks, including required attributes, parsed dates, and product pricing warnings, are enforced in the Silver layer and are not repeated here.

In [ ]:
# Validate dimension keys and obtain Gold row counts.
date_rows = validate_unique_key(dim_date_df, "dim_date", ["date_key"])
product_rows = validate_unique_key(dim_product_df, "dim_product", ["product_id"])
store_rows = validate_unique_key(dim_store_df, "dim_store", ["store_id"])

print("Dimension key and grain checks passed.")

print({
    "dim_date": date_rows,
    "dim_product": product_rows,
    "dim_store": store_rows,
})

### 3.5 Write dimensions
After the blocking key checks pass, write the dimensions as Delta tables using overwrite semantics.

In [ ]:
overwrite_delta_table(dim_date_df, "dim_date")
overwrite_delta_table(dim_product_df, "dim_product")
overwrite_delta_table(dim_store_df, "dim_store")

## 4. Build facts

### 4.1 Sales fact

Preserve one row per sale and attach the product price and cost used for this Gold build. Persisting `unit_price` and `unit_cost` in the fact prevents later dimension changes from altering already-built monetary results.

In [ ]:
product_value_lookup_df = dim_product_df.select(
    "product_id",
    F.col("product_price").alias("unit_price"),
    F.col("product_cost").alias("unit_cost"),
)

fact_sales_df = (
    silver_sales_df.alias("s")
    .join(product_value_lookup_df.alias("p"), on="product_id", how="inner")
    .select(
        F.col("s.sale_id").alias("sale_id"),
        F.date_format(F.col("s.sale_date"), "yyyyMMdd").cast("int").alias("date_key"),
        F.col("product_id"),
        F.col("s.store_id").alias("store_id"),
        F.col("s.units").alias("units"),
        F.col("p.unit_price").cast("decimal(18,2)").alias("unit_price"),
        F.col("p.unit_cost").cast("decimal(18,2)").alias("unit_cost"),
    )
    .withColumn("sales_amount", (F.col("units") * F.col("unit_price")).cast("decimal(18,2)"))
    .withColumn("cost_amount", (F.col("units") * F.col("unit_cost")).cast("decimal(18,2)"))
    .withColumn("gross_profit", (F.col("sales_amount") - F.col("cost_amount")).cast("decimal(18,2)"))
)

display(fact_sales_df.orderBy("sale_id").limit(10))

### 4.2 Inventory-position fact

Preserve only supplied store-product positions and calculate their cost and retail values. `source_ingested_at` is audit metadata, not a business snapshot date.

In [ ]:
fact_inventory_position_df = (
    silver_inventory_df.alias("i")
    .join(product_value_lookup_df.alias("p"), on="product_id", how="inner")
    .select(
        F.col("i.store_id").alias("store_id"),
        F.col("product_id"),
        F.col("i.stock_on_hand").alias("stock_on_hand"),
        (F.col("i.stock_on_hand") * F.col("p.unit_cost")).cast("decimal(18,2)").alias("inventory_cost_value"),
        (F.col("i.stock_on_hand") * F.col("p.unit_price")).cast("decimal(18,2)").alias("inventory_retail_value"),
        F.col("i.ingested_at").alias("source_ingested_at"),
    )
)

display(fact_inventory_position_df.orderBy("store_id", "product_id").limit(10))

### 4.3 Validate facts

Foreign-key checks use the original Silver facts before the inner joins so unmatched records cannot disappear silently. Additional blocking checks enforce fact grain, valid quantities, complete monetary values, and fixed-dataset reconciliation.

In [ ]:
# Validate the declared fact grains.

sales_rows = validate_unique_key(
    fact_sales_df,
    "fact_sales",
    ["sale_id"],
)

inventory_rows = validate_unique_key(
    fact_inventory_position_df,
    "fact_inventory_position",
    ["store_id", "product_id"],
)

In [ ]:
# Confirm that dimension lookups did not lose or duplicate fact rows.

silver_sales_rows = silver_sales_df.count()
silver_inventory_rows = silver_inventory_df.count()

require(
    sales_rows == silver_sales_rows,
    (
        "fact_sales row count does not match silver_sales after "
        f"dimension lookups: Gold={sales_rows}, "
        f"Silver={silver_sales_rows}"
    ),
)

require(
    inventory_rows == silver_inventory_rows,
    (
        "fact_inventory_position row count does not match "
        f"silver_inventory_position: Gold={inventory_rows}, "
        f"Silver={silver_inventory_rows}"
    ),
)

In [ ]:
# Validate the new Gold monetary calculations.

invalid_sales_amounts = fact_sales_df.filter(
    F.col("sales_amount").isNull()
    | F.col("cost_amount").isNull()
    | F.col("gross_profit").isNull()
).count()

require(
    invalid_sales_amounts == 0,
    (
        "fact_sales contains "
        f"{invalid_sales_amounts} rows with missing calculated amounts."
    ),
)

invalid_inventory_amounts = fact_inventory_position_df.filter(
    F.col("inventory_cost_value").isNull()
    | F.col("inventory_retail_value").isNull()
).count()

require(
    invalid_inventory_amounts == 0,
    (
        "fact_inventory_position contains "
        f"{invalid_inventory_amounts} rows with missing calculated amounts."
    ),
)


In [ ]:
# Confirm that the sales aggregation was preserved.

silver_total_units = silver_sales_df.agg(
    F.sum("units").alias("total_units")
).first()["total_units"]

gold_total_units = fact_sales_df.agg(
    F.sum("units").alias("total_units")
).first()["total_units"]

require(
    gold_total_units == silver_total_units,
    (
        "Gold total units do not match Silver: "
        f"Gold={gold_total_units}, Silver={silver_total_units}"
    ),
)

print("Fact grain, lookup, and calculation checks passed.")

print({
    "fact_sales_rows": sales_rows,
    "fact_inventory_position_rows": inventory_rows,
    "total_units": gold_total_units,
})

### 4.4 Write facts

Overwrite the two Gold fact tables only after all fact-level blocking checks pass.

In [ ]:
overwrite_delta_table(fact_sales_df, "fact_sales")
overwrite_delta_table(fact_inventory_position_df, "fact_inventory_position")

## 5. Gold summary

Display the row counts of the persisted Gold tables after the write operations complete.

This summary is provided for execution monitoring and documentation. Source-data quality checks are enforced in the Silver layer, while Gold-specific grain and join checks are performed before writing.

In [ ]:
gold_summary = [
    ("dim_date", spark.table("dim_date").count()),
    ("dim_product", spark.table("dim_product").count()),
    ("dim_store", spark.table("dim_store").count()),
    ("fact_sales", spark.table("fact_sales").count()),
    (
        "fact_inventory_position",
        spark.table("fact_inventory_position").count(),
    ),
]

gold_summary_df = spark.createDataFrame(
    gold_summary,
    ["table_name", "row_count"],
)

display(gold_summary_df)

print("Gold build completed successfully.")

## 6. Rerun verification

Run this notebook a second time without changing or reprocessing Silver. The five saved row counts, total units, and displayed monetary totals must remain unchanged. This demonstrates deterministic and idempotent Gold reconstruction from the same Silver state.